# 06 — Contact-Rate Assumptions

## 1. Introduction and objectives

Olist records orders, not customer contacts. To build a contact center around Olist, CallVerse must **assume** how often each order situation leads a customer to contact support, about which intent, and when. This notebook (Phase 1E-c):

1. Records the Phase 1E-b decisions.
2. Loads the contact assumptions from `configs/data.yaml` and checks that they are consistent.
3. Assigns each order in the analysis window to its **situations** (late, never delivered, canceled, …).
4. Checks that each timing rule is feasible for the orders it applies to.
5. Computes the **expected** number of contacts per situation, intent and month under low, base and high scenarios.
6. Compares the base scenario with published benchmarks.

No contact is generated here. The generator (Phase 1E-e) will draw random contacts using exactly these assumptions.

> **Every number in this notebook that concerns contacts is an assumption, not an observation.** The assumptions live in one config file so they can be reviewed, changed and re-run.

## 2. Decisions from Phase 1E-b

| Question | Decision |
|---|---|
| Intent scope | Accept the proposal: the **17 order-linked** Bitext intents and their Olist signals |
| General intents (account, website, policies) | **Left out** in version 1 |
| Language | **English** contact center; Bitext messages used as-is |
| Offensive messages | **Filtered out for demos** by default; a flag keeps them available for robustness tests |

Version 1 also assumes that an order contacts support **at most once per situation**. An order in two situations, for example delivered late *and* reviewed badly, can produce two contacts.

## 3. Imports, paths and inputs

In [1]:
from pathlib import Path

import pandas as pd
import yaml

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CONFIG_FILE = PROJECT_ROOT / "configs" / "data.yaml"

required_files = [
    CONFIG_FILE,
    PROCESSED_DIR / "olist_orders_analytical.parquet",
    PROCESSED_DIR / "bitext_intent_catalog.csv",
]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        f"Missing inputs: {missing_files}. Run notebooks 03, 04 and 05 first."
    )

with open(CONFIG_FILE, encoding="utf-8") as config_file:
    config = yaml.safe_load(config_file)
contact_config = config["generation"]["contacts"]

orders = pd.read_parquet(PROCESSED_DIR / "olist_orders_analytical.parquet")
intent_catalog = pd.read_csv(PROCESSED_DIR / "bitext_intent_catalog.csv")
window_orders = orders[orders["in_analysis_window"]].copy()

print("Config version:", contact_config["version"])
print("Decisions:", contact_config["decisions"])
print("Orders in the analysis window:", len(window_orders))
print("Situations in the config:", list(contact_config["situations"]))

Config version: 1
Decisions: {'intent_scope': 'order_linked_only', 'language': 'english', 'offensive_messages': 'filter_for_demos', 'max_contacts_per_situation': 1}
Orders in the analysis window: 99092
Situations in the config: ['overdue_never_delivered', 'delivered_late', 'seller_shipped_late', 'on_time_tracking_check', 'canceled_or_unavailable', 'post_delivery_problem', 'payment_anomaly', 'change_before_shipping', 'invoice_request']


## 4. Check the assumptions

Before using the config we check that:

- every probability is between 0 and 1;
- each situation's intent split adds up to 1;
- every intent used is one of the 17 order-linked intents from the Phase 1E-a catalog;
- every order-linked intent is used by at least one situation;
- every timing anchor is a column of the order table.

In [2]:
situations_config = contact_config["situations"]
order_linked_intents = set(
    intent_catalog.loc[intent_catalog["proposed_scope"] == "order_linked", "intent"]
)

config_rows = []
problems = []
used_intents = set()
for situation_name, settings in situations_config.items():
    probability = settings["probability"]
    intent_split = settings["intents"]
    timing = settings["timing"]
    split_total = round(sum(intent_split.values()), 6)
    used_intents.update(intent_split)

    if not 0 <= probability <= 1:
        problems.append(
            f"{situation_name}: probability {probability} is not between 0 and 1"
        )
    if split_total != 1:
        problems.append(
            f"{situation_name}: intent split adds up to {split_total}, not 1"
        )
    for intent in intent_split:
        if intent not in order_linked_intents:
            problems.append(
                f"{situation_name}: intent '{intent}' is not an order-linked intent"
            )
    for column_key in ["anchor", "before"]:
        column = timing.get(column_key)
        if column is not None and column not in orders.columns:
            problems.append(
                f"{situation_name}: timing column '{column}' not in the order table"
            )

    config_rows.append(
        {
            "situation": situation_name,
            "probability": probability,
            "intents": len(intent_split),
            "anchor": timing["anchor"],
            "days": f"{timing['min_days']}-{timing['max_days']}",
            "must_be_before": timing.get("before", ""),
        }
    )

unused_intents = sorted(order_linked_intents - used_intents)
if unused_intents:
    problems.append(f"Order-linked intents used by no situation: {unused_intents}")

display(pd.DataFrame(config_rows))
if problems:
    raise ValueError("Config problems:\n" + "\n".join(problems))
print(
    "Config checks passed: probabilities valid, splits add up to 1, all 17 order-linked intents covered."
)

,situation,probability,intents,anchor,days,must_be_before
0,overdue_never_delivered,0.600,4,order_estimated_delivery_date,1-7,
1,delivered_late,0.350,4,order_estimated_delivery_date,1-3,order_delivered_customer_date
2,seller_shipped_late,0.050,4,latest_shipping_limit_date,1-3,order_delivered_carrier_date
3,on_time_tracking_check,0.015,3,order_delivered_carrier_date,0-5,order_delivered_customer_date
4,canceled_or_unavailable,0.500,3,order_purchase_timestamp,1-7,
5,post_delivery_problem,0.300,7,order_delivered_customer_date,0-7,
6,payment_anomaly,0.300,1,order_purchase_timestamp,0-2,
7,change_before_shipping,0.005,1,order_purchase_timestamp,0-1,order_delivered_carrier_date
8,invoice_request,0.005,1,order_purchase_timestamp,0-14,


Config checks passed: probabilities valid, splits add up to 1, all 17 order-linked intents covered.


## 5. Assign orders to situations

Each situation is a yes/no rule on the Phase 1D order table. The four delivery situations are **mutually exclusive**, so one order produces at most one "where is my order?" contact:

| Situation | Rule |
|---|---|
| `overdue_never_delivered` | `overdue_without_delivery` |
| `delivered_late` | delivered after the estimated date |
| `seller_shipped_late` | delivered on time, but the seller handed over after the shipping deadline |
| `on_time_tracking_check` | delivered on time, seller on time |
| `canceled_or_unavailable` | final status canceled or unavailable |
| `post_delivery_problem` | delivered, latest review 1–2 stars with a comment |
| `payment_anomaly` | payment differs from order value by more than 1 BRL, or type `not_defined` |
| `change_before_shipping` | any order that is not canceled or unavailable |
| `invoice_request` | any order with a payment |

In [3]:
# fillna(False): an unknown value (<NA>) does not put an order in a situation
delivered = window_orders["has_customer_delivery_timestamp"]
late = window_orders["delivered_after_estimated_date"].fillna(False)
seller_late = window_orders["carrier_after_shipping_limit"].fillna(False)
overdue = window_orders["overdue_without_delivery"]
canceled_or_unavailable = window_orders["is_canceled"] | window_orders["is_unavailable"]
low_review_with_comment = window_orders["review_is_low_score"].fillna(
    False
) & window_orders["review_has_message"].fillna(False)
payment_anomaly = (window_orders["payment_minus_order_value"].abs() > 1) | (
    window_orders["primary_payment_type"] == "not_defined"
)

situation_masks = {
    "overdue_never_delivered": overdue,
    "delivered_late": delivered & late,
    "seller_shipped_late": delivered & ~late & seller_late,
    "on_time_tracking_check": delivered & ~late & ~seller_late,
    "canceled_or_unavailable": canceled_or_unavailable,
    "post_delivery_problem": delivered & low_review_with_comment,
    "payment_anomaly": payment_anomaly.fillna(False),
    "change_before_shipping": ~canceled_or_unavailable,
    "invoice_request": window_orders["has_payments"],
}

if set(situation_masks) != set(situations_config):
    raise ValueError("Situations in the notebook and in configs/data.yaml must match.")

situation_counts = []
for situation_name, mask in situation_masks.items():
    situation_counts.append(
        {
            "situation": situation_name,
            "orders": int(mask.sum()),
            "percent_of_window": round(mask.mean() * 100, 2),
        }
    )
display(pd.DataFrame(situation_counts))

# Check that the four delivery situations never overlap
delivery_situations = [
    "overdue_never_delivered",
    "delivered_late",
    "seller_shipped_late",
    "on_time_tracking_check",
]
delivery_overlap = sum(
    situation_masks[name].astype(int) for name in delivery_situations
)
print("Orders in more than one delivery situation:", int((delivery_overlap > 1).sum()))

,situation,orders,percent_of_window
0,overdue_never_delivered,1696,1.71
1,delivered_late,6532,6.59
2,seller_shipped_late,6770,6.83
3,on_time_tracking_check,82902,83.66
4,canceled_or_unavailable,1182,1.19
5,post_delivery_problem,9194,9.28
6,payment_anomaly,246,0.25
7,change_before_shipping,97910,98.81
8,invoice_request,99092,100.00


Orders in more than one delivery situation: 0


In [4]:
# How many situations does an order belong to, excluding the two base-rate situations?
specific_situations = [
    name
    for name in situation_masks
    if name not in ["change_before_shipping", "invoice_request"]
]
situations_per_order = sum(
    situation_masks[name].astype(int) for name in specific_situations
)
display(
    situations_per_order.value_counts()
    .sort_index()
    .rename("orders")
    .rename_axis("specific_situations")
    .to_frame()
)

print(
    "Example overlap: delivered late AND post-delivery problem:",
    int(
        (
            situation_masks["delivered_late"] & situation_masks["post_delivery_problem"]
        ).sum()
    ),
)

,orders
specific_situations,
0,11
1,89666
2,9389
3,26


Example overlap: delivered late AND post-delivery problem: 2745


## 6. Timing feasibility

Each contact happens a number of days after an **anchor** event (between `min_days` and `max_days`). Some rules also require the contact to happen **before** another event; for example, a "where is my order?" contact about a late order must happen before the order arrives.

For each situation we check how many orders have at least one possible contact day. Two cases make a contact impossible:

1. The anchor date is missing.
2. The earliest possible contact day falls after the "before" event, or after the effective data end (31 August 2018).

Orders without a possible day will not receive a contact. Removing them keeps the generated contacts consistent with the order timeline.

In [5]:
EFFECTIVE_DATA_END = pd.Timestamp("2018-08-31")  # derived in notebook 04, section 10.2

feasibility_rows = []
feasible_masks = {}
for situation_name, mask in situation_masks.items():
    timing = situations_config[situation_name]["timing"]
    anchor_day = window_orders[timing["anchor"]].dt.normalize()
    earliest_day = anchor_day + pd.to_timedelta(timing["min_days"], unit="D")

    feasible = mask & anchor_day.notna() & (earliest_day <= EFFECTIVE_DATA_END)
    if "before" in timing:
        before_day = window_orders[timing["before"]].dt.normalize()
        # The contact may happen on the same day as the "before" event (e.g. the morning before delivery)
        feasible = feasible & before_day.notna() & (earliest_day <= before_day)

    feasible_masks[situation_name] = feasible
    feasibility_rows.append(
        {
            "situation": situation_name,
            "orders_in_situation": int(mask.sum()),
            "orders_with_possible_day": int(feasible.sum()),
            "percent_feasible": round(feasible.sum() / mask.sum() * 100, 2)
            if mask.sum()
            else None,
        }
    )
feasibility_df = pd.DataFrame(feasibility_rows)
display(feasibility_df)

,situation,orders_in_situation,orders_with_possible_day,percent_feasible
0,overdue_never_delivered,1696,1696,100.00
1,delivered_late,6532,6532,100.00
2,seller_shipped_late,6770,3986,58.88
3,on_time_tracking_check,82902,82897,99.99
4,canceled_or_unavailable,1182,1181,99.92
5,post_delivery_problem,9194,9169,99.73
6,payment_anomaly,246,246,100.00
7,change_before_shipping,97910,97304,99.38
8,invoice_request,99092,99092,100.00


A feasibility below 100% means some orders cannot produce a contact under the timing rule. Typical causes: an order in transit for less time than the minimum delay, a seller who handed over on the day after the deadline, or a promised date too close to the end of the data.

## 7. Expected contact volumes

The **expected** number of contacts for a situation is its probability × the number of feasible orders. The generator will produce numbers close to these, varying slightly from run to run.

The three scenarios multiply every probability by the scenario factor (low = 0.5, base = 1, high = 1.5), capped at 1.

In [6]:
scenarios = contact_config["scenarios"]

expected_rows = []
for situation_name, settings in situations_config.items():
    feasible_orders = int(feasible_masks[situation_name].sum())
    row = {
        "situation": situation_name,
        "feasible_orders": feasible_orders,
        "base_probability": settings["probability"],
    }
    for scenario_name, factor in scenarios.items():
        probability = min(settings["probability"] * factor, 1.0)
        row[f"expected_{scenario_name}"] = round(probability * feasible_orders, 1)
    expected_rows.append(row)
expected_by_situation = pd.DataFrame(expected_rows)

totals = expected_by_situation[[f"expected_{name}" for name in scenarios]].sum()
display(expected_by_situation)
print("Total expected contacts:", totals.round(0).to_dict())
print(
    "Expected contacts per 100 orders:",
    (totals / len(window_orders) * 100).round(2).to_dict(),
)

,situation,feasible_orders,base_probability,expected_low,expected_base,expected_high
0,overdue_never_delivered,1696,0.600,508.8,1017.6,1526.4
1,delivered_late,6532,0.350,1143.1,2286.2,3429.3
2,seller_shipped_late,3986,0.050,99.7,199.3,299.0
3,on_time_tracking_check,82897,0.015,621.7,1243.5,1865.2
4,canceled_or_unavailable,1181,0.500,295.2,590.5,885.8
5,post_delivery_problem,9169,0.300,1375.3,2750.7,4126.0
6,payment_anomaly,246,0.300,36.9,73.8,110.7
7,change_before_shipping,97304,0.005,243.3,486.5,729.8
8,invoice_request,99092,0.005,247.7,495.5,743.2


Total expected contacts: {'expected_low': 4572.0, 'expected_base': 9144.0, 'expected_high': 13715.0}
Expected contacts per 100 orders: {'expected_low': 4.61, 'expected_base': 9.23, 'expected_high': 13.84}


### 7.1 Expected contacts per intent (base scenario)

In [7]:
intent_rows = []
for situation_name, settings in situations_config.items():
    expected_contacts = settings["probability"] * int(
        feasible_masks[situation_name].sum()
    )
    for intent, share in settings["intents"].items():
        intent_rows.append(
            {
                "situation": situation_name,
                "intent": intent,
                "expected_contacts": expected_contacts * share,
            }
        )
intent_expected = pd.DataFrame(intent_rows)

expected_by_intent = (
    intent_expected.groupby("intent")["expected_contacts"]
    .sum()
    .sort_values(ascending=False)
    .to_frame()
)
expected_by_intent["share_percent"] = (
    expected_by_intent["expected_contacts"]
    / expected_by_intent["expected_contacts"].sum()
    * 100
).round(2)
expected_by_intent = expected_by_intent.merge(
    intent_catalog.set_index("intent")[["category"]], left_index=True, right_index=True
)
display(expected_by_intent.round(1))

,expected_contacts,share_percent,category
intent,,,
track_order,1847.8,20.2,ORDER
track_delivery,1311.0,14.3,DELIVERY
delivery_time,887.1,9.7,DELIVERY
delivery_issue,700.6,7.7,DELIVERY
product_issue,550.1,6.0,PRODUCT
request_invoice,495.5,5.4,ORDER
change_order,486.5,5.3,ORDER
return_product,412.6,4.5,RETURNS
wrong_item,412.6,4.5,DELIVERY


## 8. Comparison with benchmarks

We compare the base scenario with published e-commerce benchmarks for **"where is my order?" (WISMO) contacts**, the intents `track_order`, `track_delivery`, `delivery_time` and `delivery_issue`. The benchmarks are vendor-published, use different definitions, and serve only as a sanity check:

- WISMO contacts are typically **3–10% of orders** (3–5% is often called healthy).
- WISMO is often quoted at about **25–40% of all contacts**. Our version-1 scope leaves out general questions, so WISMO will take a larger share of *our* contacts than of a full contact center's.

In [8]:
WISMO_INTENTS = ["track_order", "track_delivery", "delivery_time", "delivery_issue"]
benchmarks = contact_config["benchmarks"]

benchmark_rows = []
for scenario_name, factor in scenarios.items():
    wismo_contacts = 0.0
    all_contacts = 0.0
    for situation_name, settings in situations_config.items():
        probability = min(settings["probability"] * factor, 1.0)
        expected_contacts = probability * int(feasible_masks[situation_name].sum())
        all_contacts += expected_contacts
        for intent, share in settings["intents"].items():
            if intent in WISMO_INTENTS:
                wismo_contacts += expected_contacts * share
    wismo_rate = wismo_contacts / len(window_orders)
    benchmark_rows.append(
        {
            "scenario": scenario_name,
            "wismo_contacts": round(wismo_contacts),
            "wismo_rate_percent": round(wismo_rate * 100, 2),
            "within_benchmark_3_to_10_percent": benchmarks["wismo_rate_low"]
            <= wismo_rate
            <= benchmarks["wismo_rate_high"],
            "wismo_share_of_our_contacts_percent": round(
                wismo_contacts / all_contacts * 100, 1
            ),
        }
    )
display(pd.DataFrame(benchmark_rows))

,scenario,wismo_contacts,wismo_rate_percent,within_benchmark_3_to_10_percent,wismo_share_of_our_contacts_percent
0,low,2373,2.40,False,51.9
1,base,4747,4.79,True,51.9
2,high,7120,7.19,True,51.9


If the base scenario's WISMO rate is inside the benchmark range, the delivery assumptions produce a plausible order of magnitude. That is only a sanity check: it does not validate the assumptions.

## 9. Expected monthly volume

To see the shape of future contact demand, we place each expected contact in the month of its **earliest possible contact day** (anchor + `min_days`). This approximates the timing; the generator will spread contacts over the full window.

In [9]:
monthly_frames = []
for situation_name, settings in situations_config.items():
    feasible = feasible_masks[situation_name]
    timing = settings["timing"]
    earliest_day = window_orders.loc[
        feasible, timing["anchor"]
    ].dt.normalize() + pd.to_timedelta(timing["min_days"], unit="D")
    monthly = (
        earliest_day.dt.strftime("%Y-%m").value_counts() * settings["probability"]
    ).rename(situation_name)
    monthly_frames.append(monthly)

monthly_expected = pd.concat(monthly_frames, axis=1).fillna(0).sort_index()
monthly_expected["total_expected_contacts"] = monthly_expected.sum(axis=1)
monthly_orders = (
    window_orders["purchase_year_month"].value_counts().rename("orders_purchased")
)
monthly_expected = monthly_expected.join(monthly_orders, how="left")
monthly_expected["contacts_per_100_orders"] = (
    monthly_expected["total_expected_contacts"]
    / monthly_expected["orders_purchased"]
    * 100
).round(2)
display(
    monthly_expected[
        [
            "orders_purchased",
            "total_expected_contacts",
            "contacts_per_100_orders",
            "delivered_late",
            "overdue_never_delivered",
            "post_delivery_problem",
        ]
    ].round(0)
)

,orders_purchased,total_expected_contacts,contacts_per_100_orders,delivered_late,overdue_never_delivered,post_delivery_problem
2017-01,800,33.0,4.0,0.0,0.0,6.0
2017-02,1780,119.0,7.0,4.0,4.0,30.0
2017-03,2682,241.0,9.0,26.0,59.0,54.0
2017-04,2404,198.0,8.0,34.0,41.0,51.0
2017-05,3700,335.0,9.0,64.0,50.0,93.0
2017-06,3245,272.0,8.0,32.0,56.0,75.0
2017-07,4026,294.0,7.0,33.0,44.0,80.0
2017-08,4331,315.0,7.0,40.0,41.0,93.0
2017-09,4285,327.0,8.0,51.0,55.0,87.0
2017-10,4631,363.0,8.0,58.0,41.0,109.0


Contacts per 100 orders are not constant. Delivery contacts happen after the promised date, so they appear *after* the periods of late deliveries: December 2017 (after the Black Friday orders of November) and March–April 2018 (after the February–March delays) have clearly more contacts per order. Because the table places contacts by contact month while orders are counted by purchase month, the ratio is an approximation. This is the behaviour the architecture's forecast formula needs, `orders(t) × contact-rate(intent | order state, promo, delay)`: contact demand depends on order *situations*, not only on order volume.

## 10. Time-of-day profile

Contacts need a time of day for the queue simulation. Olist has no contact times, so version 1 assumes contacts follow the **hourly profile of Olist purchases**, which reflects when customers are online. The table shows that profile.

In [10]:
hour_profile = (
    window_orders["purchase_hour"]
    .value_counts(normalize=True)
    .sort_index()
    .rename("share")
)
hour_profile_df = (hour_profile * 100).round(2).rename("percent").to_frame()
display(hour_profile_df.T)
print(
    "Busiest hours:",
    hour_profile.nlargest(3).index.tolist(),
    "| quietest hours:",
    hour_profile.nsmallest(3).index.tolist(),
)

purchase_hour,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
percent,2.41,1.18,0.51,0.27,0.2,0.19,0.51,1.24,2.98,4.81,...,6.61,6.48,6.72,6.19,5.81,6.03,6.23,6.26,5.83,4.15


Busiest hours: [16, 11, 14] | quietest hours: [5, 4, 3]


## 11. Export

Saved for the generator and the report:

- `contact_expected_by_situation.csv`: feasible orders and expected contacts per scenario for each situation.
- `contact_expected_by_intent.csv`: expected contacts per intent (base scenario).
- `contact_hour_profile.csv`: share of contacts per hour of day.

The assumptions themselves stay in `configs/data.yaml`, the single source of truth.

In [11]:
expected_by_situation.to_csv(
    PROCESSED_DIR / "contact_expected_by_situation.csv", index=False
)
expected_by_intent.round(2).rename_axis("intent").reset_index().to_csv(
    PROCESSED_DIR / "contact_expected_by_intent.csv", index=False
)
hour_profile.rename_axis("hour").reset_index().to_csv(
    PROCESSED_DIR / "contact_hour_profile.csv", index=False
)

for file_name in [
    "contact_expected_by_situation.csv",
    "contact_expected_by_intent.csv",
    "contact_hour_profile.csv",
]:
    reloaded = pd.read_csv(PROCESSED_DIR / file_name)
    print(f"{file_name}: {len(reloaded)} rows, {len(reloaded.columns)} columns")

contact_expected_by_situation.csv: 9 rows, 6 columns
contact_expected_by_intent.csv: 17 rows, 4 columns
contact_hour_profile.csv: 24 rows, 2 columns


## 12. Summary and limitations

**What this phase defines**

- Nine order situations, each with a contact probability, an intent split and a timing rule (`configs/data.yaml`).
- The expected contact volume they imply, per situation, intent and month, under three scenarios.
- A time-of-day profile for contacts.

**Limitations**

- **All contact parameters are assumptions.** No dataset links Olist orders to contacts. The benchmark check is a sanity check on the order of magnitude, not a validation.
- **Benchmarks are vendor-published** and use different definitions.
- **Weak signals.** Post-delivery problems are inferred from low reviews with comments; payment issues from payment mismatches.
- **One contact per situation.** Repeat contacts (calling again because the first contact did not solve the problem) are not modelled in version 1.
- **No channels yet.** Phone, chat and email are not distinguished.
- **No promotion effect yet.** The promotion calendar (Phase 1E-d) may add or scale contacts during campaign periods.
- **Hour profile.** Contacts are assumed to follow purchase hours; real contact centers also see peaks linked to opening hours.

**Next steps**

- Phase 1E-d: synthetic promotion calendar.
- Phase 1E-e: generate the synthetic contacts table (seeded, reproducible), attach a Bitext message to each contact, and label every row as synthetic.